# OT Probe — cheap directional-transport feasibility gate

**Standalone.** iMSD (STICS) reads σ²(τ) — an ISOTROPIC spread that averages over
direction, so a *coherent* drift (all nearby pixels moving the same way, e.g.
Zidovska-style chromatin motion) can partially cancel in that average and, if the net
displacement stays sub-PSF, disappear into the noise floor entirely (this is exactly
what happened in `gpu_stics.py`: `resolved=False` on this dataset). The ACF is
direction-blind too.

A transport map (real Sinkhorn OT, or the cheap FFT phase-correlation analogue used
here) reads a **vector** field per patch instead of a scalar spread — so a coherent
sub-PSF drift can still show up as a reproducible, spatially-coherent displacement even
when iMSD sees only floor. This notebook is the cheap **gate**, not real OT: it decides
whether Sinkhorn OT is worth building, without paying for it.

Three pass/fail checks (all three must pass to green-light real OT):
1. **Reproducible** — half-split (first half vs second half in time) correlation of
   per-patch transport magnitude.
2. **Independent** — transport magnitude vs CV² (density) and vs γ (ACF decay rate),
   the two established axes.
3. **Coherent** — neighbouring patches' transport *vectors* align more than a spatial
   shuffle of the same vectors would (this is what iMSD structurally cannot see).

**Paths & preprocessing mirror `iscors_real_runner.ipynb` / `chi4_probe.ipynb`
exactly** (same ZIP_PATH, binning, flat-field, BG removal → identical `video_proc`).

In [ ]:
# ── setup: clone repo (for utils/ot_probe.py + utils/gpu_iscors_fit.py) + deps ──
import os, subprocess, sys
REPO='https://github.com/breezy90126/iscors-net.git'
BRANCH='claude/ot-probe-phase-correlation-vvyq80'
REPO_DIR='/content/iscors-net'
try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
except Exception: pass
if os.path.isdir(REPO_DIR):
    subprocess.run(['git','-C',REPO_DIR,'fetch','origin'],check=False)
    subprocess.run(['git','-C',REPO_DIR,'checkout',BRANCH],check=False)
    subprocess.run(['git','-C',REPO_DIR,'pull','origin',BRANCH],check=False)
else:
    subprocess.run(['git','clone','--branch',BRANCH,REPO,REPO_DIR],check=False)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
subprocess.run(['pip','install','-q','tifffile','scipy'],check=False)
print('setup done:', os.getcwd())

In [ ]:
# ── config — SAME paths as iscors_real_runner.ipynb / chi4_probe.ipynb ───
import os, numpy as np
ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'   # raw video zip
EXTRACT_DIR = '/content/real_data'                                  # local extraction
SAVE_DIR    = '/content/drive/MyDrive/iscors_test'                  # results dir
VIDEO_FNAME = 'COBRI_rarw_video.tif'
MASK_PATH   = 'data/condensation_mask.tif'                          # cell mask (repo-local)
N_FRAMES    = 1000        # probe is cheap by design; 500-1000 frames is enough
BIN_FACTOR  = 2
CHUNK_SIZE  = 100
MIN_CV      = 0.005
TAUS        = (4, 8, 16, 32)     # transport lags (frames) to probe, confidence-averaged
PATCH       = 24                 # patch size (px, POST-binning) for phase correlation
RECON_TAUS  = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)   # for the gamma-map independence check
GAMMA_SCALE = 2.0
os.makedirs(EXTRACT_DIR, exist_ok=True); os.makedirs(SAVE_DIR, exist_ok=True)
print('ZIP_PATH :', ZIP_PATH)
print('TAUS     :', TAUS, ' PATCH:', PATCH)

In [ ]:
# ── load + preprocess (identical pipeline to chi4_probe.ipynb / p2-preprocess) ──
import zipfile, tifffile
from scipy.ndimage import gaussian_filter
def _find(root, name):
    for dp, _, fs in os.walk(root):
        if name in fs: return os.path.join(dp, name)
    return None
VIDEO_PATH = _find(EXTRACT_DIR, VIDEO_FNAME)
if not VIDEO_PATH:
    print(f'Extracting {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH,'r') as zf: zf.extractall(EXTRACT_DIR)
    VIDEO_PATH = _find(EXTRACT_DIR, VIDEO_FNAME)
assert VIDEO_PATH, f'{VIDEO_FNAME} not found under {EXTRACT_DIR}'
print('Video:', VIDEO_PATH)

frame0 = tifffile.imread(VIDEO_PATH, key=0).astype(np.float32)
H0, W0 = frame0.shape; Hb, Wb = H0//BIN_FACTOR, W0//BIN_FACTOR
with tifffile.TiffFile(VIDEO_PATH) as tf: total = len(tf.pages)
n = min(N_FRAMES, total)
print(f'Loading {n}/{total} frames, {H0}x{W0} -> {Hb}x{Wb} (bin {BIN_FACTOR}) ...')
video_raw = np.empty((n, Hb, Wb), np.float32)
for s in range(0, n, CHUNK_SIZE):
    e = min(s+CHUNK_SIZE, n)
    ch = tifffile.imread(VIDEO_PATH, key=range(s, e)).astype(np.float32)
    video_raw[s:e] = ch.reshape(e-s, Hb, BIN_FACTOR, Wb, BIN_FACTOR).mean((2,4))
    del ch
# Step 1: flat-field ; Step 2: per-frame Gaussian BG removal (same as main)
median_xy = np.median(video_raw, axis=0)
video_ff  = video_raw / (median_xy[None] + 1e-10)
video_proc = np.empty_like(video_ff)
for t in range(len(video_ff)):
    bg = gaussian_filter(video_ff[t], sigma=4)
    video_proc[t] = video_ff[t] / (bg + 1e-10)
print(f'video_proc ready: {video_proc.shape}')

In [ ]:
# ── cell mask + the two established axes (CV², gamma) for the independence check ──
import importlib, utils.gpu_iscors_fit as _gf; importlib.reload(_gf)
from utils.gpu_iscors_fit import compute_density, gpu_fit_maps

density_map, cell_mask = compute_density(video_proc, min_cv=MIN_CV)

mask_path = _find(EXTRACT_DIR, os.path.basename(MASK_PATH)) or MASK_PATH
if os.path.isfile(mask_path):
    mk = np.asarray(tifffile.imread(mask_path)).squeeze().astype(bool)
    if mk.shape != cell_mask.shape:
        from scipy.ndimage import zoom
        mk = zoom(mk.astype(np.float32), (cell_mask.shape[0]/mk.shape[0],
                                           cell_mask.shape[1]/mk.shape[1]), order=0) > 0.5
    cell_mask = cell_mask & mk
    print('cell mask loaded from', mask_path)
else:
    print('(no external cell mask found — using compute_density auto-mask only)')

fit = gpu_fit_maps(video_proc, recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                    gamma_scale=GAMMA_SCALE, min_cv=MIN_CV, verbose=False)
gamma_map = fit['gamma']
print('density_map, gamma_map ready. cell px:', int(cell_mask.sum()))

In [ ]:
# ── run the OT probe: reproducible / independent / coherent ───────────────
import importlib, utils.ot_probe as _ot; importlib.reload(_ot)
from utils.ot_probe import ot_probe

out = ot_probe(video_proc, TAUS, patch=PATCH, mask=cell_mask,
               density_map=density_map, gamma_map=gamma_map, verbose=True)

print()
print('n_valid patches :', out['n_valid'])
print('r_half          :', round(out['r_half'], 3))
print('r_density       :', round(out['r_density'], 3))
print('r_gamma         :', round(out['r_gamma'], 3))
print('coherence z     :', round(out['coherence']['z'], 2))
print('reproducible/independent/coherent ->', out['reproducible'], out['independent'], out['coherent'])

In [ ]:
# ── plot + record ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 2, figsize=(11, 10))

# (0,0) transport vector field over the density map
ax[0,0].imshow(density_map, cmap='gray', alpha=0.6,
               vmin=np.nanpercentile(density_map,2), vmax=np.nanpercentile(density_map,98))
ys, xs = out['ys'], out['xs']
Yc, Xc = np.meshgrid(ys + PATCH//2, xs + PATCH//2, indexing='ij')
V = out['valid']
ax[0,0].quiver(Xc[V], Yc[V], out['dx'][V], -out['dy'][V], out['mag'][V],
               cmap='inferno', scale_units='xy', scale=0.2)
ax[0,0].set_title('per-patch transport vectors (color=|v|)'); ax[0,0].axis('off')

# (0,1) half-split reproducibility scatter
m1, m2 = out['mag_half1'], out['mag_half2']
ok = np.isfinite(m1) & np.isfinite(m2)
ax[0,1].scatter(m1[ok], m2[ok], s=10, alpha=0.5)
ax[0,1].set_xlabel('|v| half 1'); ax[0,1].set_ylabel('|v| half 2')
ax[0,1].set_title(f"reproducibility  r={out['r_half']:.3f}")

# (1,0) independence: transport magnitude vs CV2 / gamma (patch-grid-averaged)
from utils.ot_probe import _downsample_to_patchgrid
dens_grid  = _downsample_to_patchgrid(density_map, ys, xs, PATCH)
gamma_grid = _downsample_to_patchgrid(gamma_map,   ys, xs, PATCH)
ok2 = np.isfinite(out['mag']) & np.isfinite(dens_grid)
ax[1,0].scatter(dens_grid[ok2], out['mag'][ok2], s=10, alpha=0.5, label='vs CV2')
ax[1,0].set_xlabel('CV2 (patch mean)'); ax[1,0].set_ylabel('|v|')
ax[1,0].set_title(f"independence  r(CV2)={out['r_density']:.3f}  r(gamma)={out['r_gamma']:.3f}")

# (1,1) coherence: observed vs shuffled null
coh = out['coherence']
ax[1,1].bar(['observed', 'shuffled null'], [coh['observed'], coh['shuffled_mean']],
            yerr=[0, coh['shuffled_std']], color=['tab:red','tab:gray'])
ax[1,1].set_ylabel('mean neighbour cos-similarity')
ax[1,1].set_title(f"coherence  z={coh['z']:.2f}")

verdict = ('GO: reproducible + independent + coherent -> real Sinkhorn OT worth building'
           if out['resolved'] else
           'NO-GO: at least one gate failed -> no new directional-transport axis here')
fig.suptitle('OT probe (phase-correlation transport) — ' + verdict, fontsize=12)
plt.tight_layout(); plt.show()
fig.savefig(os.path.join(SAVE_DIR,'ot_probe.png'), dpi=120, bbox_inches='tight')
with open(os.path.join(SAVE_DIR,'ot_probe.txt'),'w') as f:
    f.write('=== OT (phase-correlation transport) feasibility probe ===\n')
    f.write(f"n_valid={out['n_valid']}  r_half={out['r_half']:.3f}  "
            f"r_density={out['r_density']:.3f}  r_gamma={out['r_gamma']:.3f}\n")
    f.write(f"coherence observed={coh['observed']:.3f}  shuffled={coh['shuffled_mean']:.3f}"
            f"+/-{coh['shuffled_std']:.3f}  z={coh['z']:.2f}\n")
    f.write(f"reproducible={out['reproducible']}  independent={out['independent']}  "
            f"coherent={out['coherent']}  resolved={out['resolved']}\n")
    f.write('VERDICT: '+verdict+'\n')
print('saved:', os.path.join(SAVE_DIR,'ot_probe.png'), 'and .txt')

## How to read it

- **All three pass (GO)** — there is a reproducible, independent, spatially-coherent
  directional transport signal that neither the ACF (γ) nor CV² nor iMSD captures.
  This is the one result that justifies building real Sinkhorn OT (equal-mass measures,
  ground-cost/regularization selection) as an independent verification axis.
- **`r_half` low** — the vector field is not reproducible; it's phase-correlation noise,
  not signal. Stop here regardless of the other two.
- **`r_density`/`r_gamma` high** — the transport magnitude is just tracking where CV² or
  γ are large (e.g. brighter/faster regions correlate-shift more by construction), not a
  new axis. OT would be redundant with what's already measured.
- **coherence `z` low** — vectors point in random directions patch-to-patch; there is no
  spatially coherent drift, only per-patch noise. This is the check iMSD structurally
  cannot perform (it never looks at direction), so a low `z` here specifically means
  "not even the vector generalization of iMSD sees anything" — the honest end of the
  line, same posture as `gpu_stics.py`/`gpu_chi4.py`'s `resolved=False`.
- **Increase `N_FRAMES`/denser `TAUS`/smaller `PATCH`** only if the result is borderline —
  the whole point of this probe is to answer the go/no-go question before paying for
  real OT, not to be tuned into a positive result.